# Lab 6 — Multi-Objective Robust Decision Making (MORDM)

## Background

**MORDM** (Kasprzyk et al., 2013) is a structured methodology for finding and evaluating
robust policies under deep uncertainty. It consists of four iterative steps:

| Step | Name | Question |
|------|------|---------|
| 1 | **Problem formulation** | What are the objectives, levers, and uncertainties? |
| 2 | **Search** | What strategies are Pareto-optimal under a reference scenario? |
| 3 | **Re-evaluate under uncertainty** | How do these strategies perform across the full uncertainty space? |
| 4 | **Scenario discovery** | Under which conditions do strategies fail? How can we redesign? |

### The DPS (Direct Policy Search) lake model

In previous labs, the lake model had **100 decision levers** (one per year). Here we use
a **closed-loop** version where the anthropogenic release $a_t$ is a function of the current
pollution level $X_t$:

$$a_t = \text{clip}\!\left(\sum_{j=1}^{2} w_j \left|\frac{X_t - c_j}{r_j}\right|^3,\; 0.01,\; 0.1\right)$$

The policy is defined by **5 parameters**: $c_1, c_2$ (centres), $r_1, r_2$ (radii), $w_1$
(weight; $w_2 = 1 - w_1$). This is a **radial basis function (RBF)** policy that responds
to pollution levels in real time — more realistic than the open-loop 100-lever formulation.

### Robustness metrics

Once candidate policies are found, we re-evaluate them under deep uncertainty and compute:

1. **Signal-to-noise ratio (S/N)**: $\text{S/N} = \mu/\sigma$ (for maximised outcomes).
   A policy with high mean and low variance has high S/N — robust performance.

2. **Maximum regret**: $\text{Regret} = |\text{best possible} - \text{actual}|$ for each scenario.
   Maximum regret = worst-case difference between the policy's performance and the best
   possible outcome in that scenario.

### What you will do

Steps 1–4 of one full MORDM iteration on the lake problem.

## Step 1: Problem formulation

We use the **DPS (closed-loop) lake model** from `dps_lake_model.py`. This is a different formulation from the **intertemporal (open-loop) model** used in Labs 4–5:

| | Intertemporal (Labs 4–5) | DPS (this lab) |
|---|---|---|
| **Policy levers** | 100 annual release decisions `l0`…`l99` | 5 RBF parameters `c1, c2, r1, r2, w1` |
| **Policy type** | Open-loop — schedule fixed at time 0 | Closed-loop — release adapts to current pollution `X_t` |
| **Search space** | 100-dimensional | 5-dimensional |

The five uncertain parameters (`b`, `q`, `mean`, `stdev`, `delta`) are the same in both formulations — they describe the lake's physical dynamics, not the policy structure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from deap.tools._hypervolume import hv as deap_hv

from ema_workbench import (Model, RealParameter, ScalarOutcome,
                           MultiprocessingEvaluator, Sample,
                           ema_logging, load_archives)
from ema_workbench.analysis import parcoords, prim

ema_logging.log_to_stderr(ema_logging.INFO)

# Import the closed-loop DPS lake model
from dps_lake_model import lake_model as lake_model_func

model = Model('lakeproblem', function=lake_model_func)

# --- Uncertain parameters ---
model.uncertainties = [
    RealParameter('b',     0.1,   0.45),
    RealParameter('q',     2.0,   4.5),
    RealParameter('mean',  0.01,  0.05),
    RealParameter('stdev', 0.001, 0.005),
    RealParameter('delta', 0.93,  0.99),
]

# --- RBF policy levers (5 parameters) ---
model.levers = [
    RealParameter('c1', -2, 2),
    RealParameter('c2', -2, 2),
    RealParameter('r1',  0, 2),
    RealParameter('r2',  0, 2),
    RealParameter('w1',  0, 1),
]

# --- Outcomes with optimisation directions ---
model.outcomes = [
    ScalarOutcome('max_P',       kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome('utility',     kind=ScalarOutcome.MAXIMIZE),
    ScalarOutcome('inertia',     kind=ScalarOutcome.MAXIMIZE),
    ScalarOutcome('reliability', kind=ScalarOutcome.MAXIMIZE),
]

## Step 2: Search for Pareto-optimal candidate solutions

> **Note:** Labs 4–5 covered ε comparison and convergence on the *intertemporal* model (100-dimensional search space). The DPS model has only a **5-dimensional** search space, so the algorithm converges faster and ε choices affect the Pareto set differently. The analysis below is not repetition — it characterises convergence for *this* model.

We run ε-NSGA-II with two different ε values to see how precision affects the number of solutions. A lower ε produces more solutions but requires more NFE to converge.

We then check convergence using `ArchiveLogger` + `EpsilonProgress` + `HypervolumeMetric`.

In [ ]:
import tempfile
_tmpdir6 = tempfile.mkdtemp()

# --- Run with coarse epsilon (epsilon = 0.1) ---
with MultiprocessingEvaluator(model) as evaluator:
    results_coarse, _ = evaluator.optimize(
        nfe=5000, searchover='levers',
        epsilons=[0.1] * len(model.outcomes),
        directory=_tmpdir6, filename='tmp_coarse.tar.gz',
    )
print(f'ε=0.1:  {len(results_coarse)} solutions')

In [ ]:
# --- Run with fine epsilon (epsilon = 0.01) ---
with MultiprocessingEvaluator(model) as evaluator:
    results_fine, _ = evaluator.optimize(
        nfe=5000, searchover='levers',
        epsilons=[0.01] * len(model.outcomes),
        directory=_tmpdir6, filename='tmp_fine.tar.gz',
    )
print(f'ε=0.01: {len(results_fine)} solutions')

In [ ]:
for results, title_str in zip(
        [results_coarse, results_fine],
        [r'$\varepsilon$ = 0.1', r'$\varepsilon$ = 0.01']):
    data = results.loc[:, [o.name for o in model.outcomes]]
    lim  = parcoords.get_limits(data)
    lim.loc[0, :] = 0
    pc = parcoords.ParallelAxes(lim)
    pc.plot(data)
    pc.invert_axis('max_P')
    plt.title(f'{title_str} — {len(results)} solutions  (5 000 NFE, DPS model)')
    plt.show()


### Convergence analysis

> **Why repeat convergence analysis here?** Labs 4–5 established convergence for the *intertemporal* model — results from that search space do not carry over. The DPS model's 5-dimensional lever space means the algorithm typically converges with far fewer NFE, and the convergence trajectory will look different. We run this check to confirm convergence *for the DPS formulation specifically* before trusting the Pareto set in steps 3–4.

We run a longer optimisation (100 000 NFE) and track convergence with two complementary indicators:

- **ε-progress**: counts how many times a new Pareto-grid cell is discovered. A plateau = convergence.
- **Hypervolume (DEAP)**: measures the volume of objective space dominated by the Pareto set relative to a reference point. A plateau = convergence. We use DEAP's `hv.hypervolume` directly — this avoids the Platypus `HypervolumeMetric` which requires non-zero range on every axis (inertia and reliability are near-constant under the reference scenario). Maximisation objectives are negated to convert the problem to minimisation for the hypervolume computation.

In [ ]:
import os, warnings

os.environ['PYTHONWARNINGS'] = 'ignore::FutureWarning'
warnings.filterwarnings("ignore", category=FutureWarning)

os.makedirs('./archives', exist_ok=True)
if os.path.exists('./archives/mordm_100k.tar.gz'):
    os.remove('./archives/mordm_100k.tar.gz')

# Epsilons sized to the observed Pareto-front ranges (from archive inspection):
#   max_P       range ≈ [0.10, 0.35] → ε = 0.01
#   utility     range ≈ [0.90, 2.60] → ε = 0.05
#   inertia     range ≈ [0.98, 1.00] → ε = 0.005  (very narrow — near-constant
#   reliability range ≈ [0.99, 1.00] → ε = 0.005   under reference scenario)
#
# Note: inertia and reliability are near-constant here because the DPS closed-loop
# policy trivially satisfies them under the single reference scenario used for
# optimisation. Their variation emerges in Step 3 when re-evaluated under the
# full uncertainty space — which is exactly why MORDM re-evaluates under uncertainty.
EPSILONS = [0.01, 0.05, 0.005, 0.005]

with MultiprocessingEvaluator(model) as evaluator:
    results, convergence = evaluator.optimize(
        nfe=100000, searchover='levers',
        epsilons=EPSILONS,
        filename='mordm_100k.tar.gz',
        directory='./archives',
    )

print(f'Final Pareto set: {len(results)} solutions')


In [ ]:
# --- Convergence: Hypervolume (DEAP) + ε-progress ---
#
# We use DEAP's hv.hypervolume instead of ema_workbench's HypervolumeMetric
# (which wraps Platypus and raises PlatypusError when any objective axis has
# zero range — which happens here because inertia and reliability are
# near-constant under the reference scenario).
#
# DEAP hv.hypervolume works on a pure minimisation formulation:
#   - max_P       → keep as-is (already minimise)
#   - utility     → negate  (maximise → minimise)
#   - inertia     → negate
#   - reliability → negate
#
# The reference point must be strictly worse (larger) than every solution
# on every axis after negation.
OBJ_NAMES = [o.name for o in model.outcomes]
NEGATE     = [False, True, True, True]   # True = maximise → negate
# Reference point: comfortably outside the observed Pareto-front ranges
REF_POINT  = [0.5, -0.0, -0.90, -0.90]

archives = load_archives('./archives/mordm_100k.tar.gz')
archives.sort(key=lambda t: t[0])   # ensure ascending NFE order

hv_values, nfe_hv = [], []
for nfe, archive_df in archives:
    pts = archive_df[OBJ_NAMES].copy()
    for col, neg in zip(OBJ_NAMES, NEGATE):
        if neg:
            pts[col] = -pts[col]
    hv_val = deap_hv.hypervolume(pts.values.tolist(), REF_POINT)
    hv_values.append(hv_val)
    nfe_hv.append(nfe)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].plot(nfe_hv, hv_values)
axes[0].set_xlabel('NFE')
axes[0].set_ylabel('Hypervolume')
axes[0].set_title('Hypervolume (DEAP) — DPS lake model')
sns.despine(ax=axes[0])

axes[1].plot(convergence['nfe'], convergence['epsilon_progress'])
axes[1].set_xlabel('NFE')
axes[1].set_ylabel(r'$\varepsilon$-progress (cumulative)')
axes[1].set_title(r'$\varepsilon$-progress — DPS lake model')
sns.despine(ax=axes[1])

plt.suptitle('Convergence indicators — 100 000 NFE')
plt.tight_layout()
plt.show()

## Step 2b: Visualise the Pareto front — tradeoff analysis

We visualise the converged Pareto set on a parallel coordinates plot. Under the reference scenario, `inertia` and `reliability` are near-constant (≈ 0.99 and ≈ 1.0) — the DPS closed-loop policy satisfies them easily. The meaningful trade-off here is between `max_P` and `utility`.

> **Why keep inertia and reliability?** When re-evaluated under uncertainty in Step 3, these objectives will vary across scenarios and may become binding constraints. This is the core motivation for MORDM's re-evaluation step.

Questions to consider:
- Which solutions trade a small increase in `max_P` for a large gain in `utility`?
- Do any solutions show variation in `inertia` or `reliability`?
- Compare qualitatively to the Lab 4–5 Pareto front: does the closed-loop policy expand the attainable region on `max_P` vs `utility`?

In [ ]:
data = results.loc[:, [o.name for o in model.outcomes]]
limits = parcoords.get_limits(data)
limits.loc[0, :] = 0

paraxes = parcoords.ParallelAxes(limits)
paraxes.plot(data)
paraxes.invert_axis('max_P')
plt.title('Pareto front — 100 000 NFE (DPS lake model)')
plt.show()

## Step 3a: Filter candidate solutions by constraint

Before re-evaluating under uncertainty, we select candidate policies. Because `inertia` and `reliability` are near-constant under the reference scenario, we use `utility` and `max_P` to pick a spread of representatives: low pollution, balanced, and high utility.

In [ ]:
# Select 3 representative policies spanning the Pareto front:
#   - lowest max_P   (most pollution-averse)
#   - midpoint       (balanced compromise)
#   - highest utility (most welfare-oriented)

results_sorted = results.sort_values('max_P').reset_index(drop=True)
n = len(results_sorted)
idx = sorted({0, n // 2, n - 1})
policies_df = results_sorted.iloc[idx].drop(columns=[o.name for o in model.outcomes])
print(f'{len(policies_df)} candidate policies selected:')
print(results_sorted.iloc[idx][['max_P', 'utility', 'inertia', 'reliability']].round(3).to_string())


In [ ]:
# Convert each row to an EMA Workbench Sample object
from ema_workbench import Sample

policies_to_evaluate = [
    Sample(str(idx), **row.to_dict())
    for idx, row in policies_df.iterrows()
]
print(f'Policies to evaluate: {len(policies_to_evaluate)}')

## Step 3b: Re-evaluate candidate policies under deep uncertainty

We run 1 000 scenarios for each of the candidate policies. This generates a dataset
that shows how each policy performs across the full deep-uncertainty space.

In [ ]:
n_scenarios = 1000

with MultiprocessingEvaluator(model) as evaluator:
    results_robust = evaluator.perform_experiments(n_scenarios, policies_to_evaluate)

experiments, outcomes_robust = results_robust
print(f'Total experiments: {experiments.shape[0]}')
print(f'Policies evaluated: {experiments["policy"].nunique()}')

## Step 4a: Robustness metric — Signal-to-Noise ratio

The **signal-to-noise (S/N) ratio** summarises a policy's performance distribution:
- For outcomes to **maximise**: $\text{S/N} = \mu / \sigma$ (high mean, low variance = robust).
- For outcomes to **minimise**: $\text{S/N} = \mu \cdot \sigma$ (low mean, low variance = robust).

We calculate S/N for each policy and outcome, then visualise on a parallel coordinates plot.

In [ ]:
def s_to_n(data, direction):
    """Signal-to-noise ratio. Higher is better for all outcomes."""
    mean = np.mean(data)
    std  = np.std(data)
    if std == 0:
        std = 1
    return mean / std if direction == ScalarOutcome.MAXIMIZE else mean * std


# Calculate S/N for each policy × outcome
overall_scores = {}
for policy_name in np.unique(experiments['policy']):
    scores = {}
    mask = experiments['policy'] == policy_name
    for outcome in model.outcomes:
        values = outcomes_robust[outcome.name][mask]
        scores[outcome.name] = s_to_n(values, outcome.kind)
    overall_scores[policy_name] = scores

scores_df = pd.DataFrame.from_dict(overall_scores).T
print(scores_df)

In [ ]:
limits_sn = parcoords.get_limits(scores_df)
limits_sn.loc[0, :] = 0

paraxes_sn = parcoords.ParallelAxes(limits_sn)
paraxes_sn.plot(scores_df)
paraxes_sn.invert_axis('max_P')
plt.title('Signal-to-noise robustness scores — candidate policies')
plt.show()

## Step 4b: Robustness metric — Maximum Regret

**Regret** for policy $p$ under scenario $s$ is:

$$\text{Regret}_{p,s} = |\text{best}(s) - y_{p,s}|$$

where $\text{best}(s) = \max_{p'} y_{p',s}$ (for maximisation outcomes).

**Maximum regret** = $\max_s \text{Regret}_{p,s}$ — the worst-case performance gap.
A policy with low maximum regret performs close to the best possible in every scenario.

In [ ]:
def calculate_regret(data, best):
    return np.abs(best - data)


# Compute regret for each outcome and policy
overall_regret = {}
max_regret     = {}

for outcome in model.outcomes:
    # Long-form DataFrame: one row per (scenario, policy) combination
    data = pd.DataFrame({
        outcome.name: outcomes_robust[outcome.name],
        'policy':     experiments['policy'],
        'scenario':   experiments['scenario'],
    })

    # Pivot to wide format: rows = scenarios, columns = policies
    data = data.pivot(index='scenario', columns='policy')[outcome.name]

    # Regret = |row-max - actual value| (works for both min and max objectives
    # because we take absolute value)
    outcome_regret = (data.max(axis=1).values[:, np.newaxis] - data).abs()

    overall_regret[outcome.name] = outcome_regret
    max_regret[outcome.name]     = outcome_regret.max()

max_regret_df = pd.DataFrame(max_regret)
print(max_regret_df)

In [ ]:
# Visualise maximum regret on a heatmap (normalised by column maximum)
fig, ax = plt.subplots(figsize=(6, max(2, len(max_regret_df) * 0.8)))
sns.heatmap(max_regret_df / max_regret_df.max(), cmap='viridis',
            annot=True, fmt='.2f', ax=ax, vmin=0, vmax=1)
ax.set_title('Maximum regret (normalised) — candidate policies')
plt.tight_layout()
plt.show()

In [ ]:
# Parallel coordinates of maximum regret
colors = sns.color_palette(n_colors=len(max_regret_df))
limits_mr = parcoords.get_limits(max_regret_df)
limits_mr.loc[0, :] = 0

paraxes_mr = parcoords.ParallelAxes(limits_mr)
for i, (idx, row) in enumerate(max_regret_df.iterrows()):
    paraxes_mr.plot(row.to_frame().T, label=str(idx), color=colors[i])
paraxes_mr.invert_axis('max_P')
paraxes_mr.legend()
plt.title('Maximum regret parallel coordinates')
plt.show()

### Regret distributions

Maximum regret summarises the *worst case*. To see the full distribution, we plot boxplots
of regret across all scenarios for each policy and outcome.

In [ ]:
# Reshape regret for box plotting
policy_regret = defaultdict(dict)
for outcome_name, regret_df in overall_regret.items():
    for policy_name in regret_df.columns:
        policy_regret[policy_name][outcome_name] = regret_df[policy_name]

fig, axes = plt.subplots(ncols=len(policies_to_evaluate),
                         figsize=(5 * len(policies_to_evaluate), 4),
                         sharey=True, sharex=True)

# Handle single-policy case
if len(policies_to_evaluate) == 1:
    axes = [axes]

for ax, (policy_name, regret) in zip(axes, policy_regret.items()):
    regret_df_plot = pd.DataFrame(regret)
    # Normalise by the column maximum for fair comparison
    regret_df_plot = regret_df_plot / max_regret_df.max(axis=0)
    sns.boxplot(data=regret_df_plot, ax=ax)
    sns.despine(ax=ax)
    ax.set_title(f'Policy {policy_name}')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Regret distributions (normalised) — by policy and outcome', y=1.02)
plt.tight_layout()
plt.show()

## Step 4c: Scenario discovery on failing scenarios

We now apply PRIM to understand **under which uncertain conditions** policies perform poorly.
This identifies the vulnerabilities of each policy and provides direction for redesign.

We focus on low utility (< 0.5) as the failure criterion. Adjust the threshold if no
failures exist.

In [ ]:
# Drop policy and lever columns — keep only the 5 uncertainty parameters
lever_cols_dps = [l.name for l in model.levers]
x = experiments.drop(columns=['policy'] + lever_cols_dps)

# Failure criterion: low utility
threshold_utility = 0.5
y_fail = outcomes_robust['utility'] < threshold_utility

print(f'Failure threshold (utility < {threshold_utility}): '
      f'{y_fail.sum()} failures out of {len(y_fail)} ({100*y_fail.mean():.1f}%)')

# Relax if too few failures for PRIM
if y_fail.sum() < 10:
    threshold_utility = 0.7
    y_fail = outcomes_robust['utility'] < threshold_utility
    print(f'Relaxed to < {threshold_utility}: {y_fail.sum()} failures')

prim_alg = prim.Prim(x, y_fail)
box = prim_alg.find_box()

In [ ]:
# box.inspect_tradeoff() uses Altair, which is not compatible with Python 3.14.
# We plot the peeling trajectory directly with matplotlib instead.
traj = box.peeling_trajectory.copy()

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(traj['coverage'], traj['density'],
                c=traj.index, cmap='viridis', s=60, zorder=3)
for i, row in traj.iterrows():
    if i % 5 == 0 or i == len(traj) - 1:
        ax.annotate(str(i), (row['coverage'], row['density']),
                    textcoords='offset points', xytext=(4, 4), fontsize=8)
plt.colorbar(sc, ax=ax, label='Peeling step')
ax.set_xlabel('Coverage')
ax.set_ylabel('Density')
ax.set_title('PRIM trade-off — coverage vs density\n(annotated numbers = box index for inspect())')
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Select a box index from the trade-off plot above (knee of coverage/density curve)
box_index = min(42, len(box.peeling_trajectory) - 1)

# style='table' prints stats to stdout; style='graph' shows a visual of box limits
print(f'--- Box {box_index} stats (table) ---')
box.inspect(box_index, style='table')

print(f'\n--- Box {box_index} limits (graph) ---')
box.inspect(box_index, style='graph')
plt.show()

## Summary — MORDM cycle review

This lab demonstrated one full iteration of the MORDM cycle:

| Step | What we did | Key finding |
|------|-------------|-------------|
| **1. Problem formulation** | DPS lake model with 5 RBF levers | Closed-loop policy responds to pollution levels |
| **2. Search** | ε-NSGA-II, 100 000 NFE | Found Pareto front; confirmed (near-)convergence |
| **3. Re-evaluation** | 1 000 scenarios × selected policies | Performance varies significantly across uncertainty space |
| **4. Robustness** | Signal-to-noise + maximum regret | Trade-off between robustness on max_P/reliability vs utility/inertia |
| **4. Scenario discovery** | PRIM on low-utility failures | `delta` (discount rate) identified as primary vulnerability driver |

### Next iteration

MORDM is iterative. Having identified that low `delta` causes poor utility performance,
the next step would be to:
1. Understand *why* low delta drives poor utility (back to the model structure).
2. Potentially expand the solution space (e.g., add lake treatment levers).
3. Re-run the MOEA with the revised problem formulation.

This cycle continues until a satisfactory set of robust solutions is found.